<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 11 - Métricas y KPIs

**Módulo:** Sistemas de Big Data (SBD - 5074)

**Contenido:** Diferencia métricas vs KPIs, Framework SMART, implementación

---

## Objetivos
- Diferenciar entre métricas y KPIs
- Diseñar KPIs SMART
- Implementar sistema de KPIs automatizado

## Métricas vs KPIs

| Aspecto | Métrica | KPI |
|---------|---------|-----|
| Cantidad | Muchas (cientos) | Pocas (3-7 por área) |
| Objetivo | Informativa | Ligada a objetivo estratégico |
| Accionable | No necesariamente | Siempre accionable |
| Ejemplo | "Visitas web: 10,000" | "Conversión: 3.5% (objetivo: 4%)" |

## 1. Sistema de gestión de KPIs

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

class KPIManager:
    """Gestor de KPIs con alertas y tendencias"""
    
    def __init__(self):
        self.kpis = {}
    
    def add_kpi(self, nombre, formula, objetivo, unidad='', alerta_rojo=None, alerta_amarillo=None):
        """Añade un nuevo KPI al sistema"""
        self.kpis[nombre] = {
            'formula': formula,
            'objetivo': objetivo,
            'unidad': unidad,
            'alerta_rojo': alerta_rojo or objetivo * 0.7,
            'alerta_amarillo': alerta_amarillo or objetivo * 0.9,
            'historial': []
        }
    
    def update_kpi(self, nombre, valor, fecha=None):
        """Actualiza el valor de un KPI"""
        fecha = fecha or datetime.now()
        self.kpis[nombre]['historial'].append({
            'fecha': fecha,
            'valor': valor
        })
    
    def get_status(self, nombre):
        """Obtiene el estado del KPI (rojo/amarillo/verde)"""
        kpi = self.kpis[nombre]
        if not kpi['historial']:
            return 'gray', 'Sin datos'
        
        valor_actual = kpi['historial'][-1]['valor']
        
        if valor_actual >= kpi['objetivo']:
            return 'green', '✅ En objetivo'
        elif valor_actual >= kpi['alerta_amarillo']:
            return 'orange', '⚠️ Atención'
        else:
            return 'red', '🚨 Crítico'
    
    def get_trend(self, nombre, periodos=5):
        """Calcula la tendencia del KPI"""
        kpi = self.kpis[nombre]
        if len(kpi['historial']) < 2:
            return 0, 'Sin tendencia'
        
        valores = [h['valor'] for h in kpi['historial'][-periodos:]]
        tendencia = (valores[-1] - valores[0]) / valores[0] * 100 if valores[0] != 0 else 0
        
        if tendencia > 5:
            return tendencia, '📈 Subiendo'
        elif tendencia < -5:
            return tendencia, '📉 Bajando'
        else:
            return tendencia, '➡️ Estable'
    
    def visualize_kpi(self, nombre):
        """Crea visualización del KPI"""
        kpi = self.kpis[nombre]
        color, estado = self.get_status(nombre)
        
        if not kpi['historial']:
            print(f"No hay datos para {nombre}")
            return None
        
        valor_actual = kpi['historial'][-1]['valor']
        
        fig = go.Figure()
        
        # Gauge principal
        fig.add_trace(go.Indicator(
            mode="gauge+number+delta",
            value=valor_actual,
            title={'text': f"{nombre}<br><span style='font-size:0.8em'>{estado}</span>"},
            delta={'reference': kpi['objetivo']},
            gauge={
                'axis': {'range': [0, kpi['objetivo'] * 1.5]},
                'bar': {'color': color},
                'steps': [
                    {'range': [0, kpi['alerta_rojo']], 'color': 'lightcoral'},
                    {'range': [kpi['alerta_rojo'], kpi['alerta_amarillo']], 'color': 'lightyellow'},
                    {'range': [kpi['alerta_amarillo'], kpi['objetivo'] * 1.5], 'color': 'lightgreen'}
                ],
                'threshold': {
                    'line': {'color': 'black', 'width': 4},
                    'thickness': 0.75,
                    'value': kpi['objetivo']
                }
            },
            number={'suffix': kpi['unidad']}
        ))
        
        fig.update_layout(height=300)
        return fig

## 2. Ejemplo de uso del sistema

In [ ]:
# Crear gestor de KPIs
manager = KPIManager()

# Definir KPIs
manager.add_kpi('Tasa Conversión', 'compras/visitas*100', objetivo=4.0, unidad='%')
manager.add_kpi('Ticket Medio', 'ventas/pedidos', objetivo=85, unidad='€')
manager.add_kpi('NPS', 'promotores-detractores', objetivo=50, unidad='pts')

# Simular datos históricos
np.random.seed(42)
for i in range(10):
    fecha = datetime.now() - timedelta(days=10-i)
    manager.update_kpi('Tasa Conversión', 3.0 + np.random.random() * 1.5, fecha)
    manager.update_kpi('Ticket Medio', 70 + np.random.random() * 20, fecha)
    manager.update_kpi('NPS', 40 + np.random.random() * 15, fecha)

# Mostrar estado de todos los KPIs
print("=== ESTADO DE KPIs ===")
for nombre in manager.kpis:
    color, estado = manager.get_status(nombre)
    tendencia, texto = manager.get_trend(nombre)
    valor = manager.kpis[nombre]['historial'][-1]['valor']
    objetivo = manager.kpis[nombre]['objetivo']
    unidad = manager.kpis[nombre]['unidad']
    print(f"\n{nombre}:")
    print(f"  Valor actual: {valor:.1f}{unidad} (objetivo: {objetivo}{unidad})")
    print(f"  Estado: {estado}")
    print(f"  Tendencia: {texto} ({tendencia:+.1f}%)")

In [ ]:
# Visualizar KPIs
for nombre in manager.kpis:
    fig = manager.visualize_kpi(nombre)
    if fig:
        fig.show()

## Framework SMART para KPIs

| Criterio | Significado | Pregunta |
|----------|-------------|----------|
| **S**pecific | Específico | ¿Qué exactamente medimos? |
| **M**easurable | Medible | ¿Cómo lo cuantificamos? |
| **A**chievable | Alcanzable | ¿Es realista? |
| **R**elevant | Relevante | ¿Importa para el negocio? |
| **T**ime-bound | Temporal | ¿En qué plazo? |

---

## Recursos adicionales

- [KPI Library](https://www.kpilibrary.com/)
- [Balanced Scorecard](https://balancedscorecard.org/)
- [OKR Examples](https://www.whatmatters.com/get-examples)